# Pre-Exposure Period Validation

**Window:** 2026-04-13 → 2026-04-22 (10 days, system 900 — system 939 only joined on Apr 21 so it cannot anchor a baseline by itself).

## Why this notebook exists

Before we trust *any* effect of the 5G exposure in the `5g_foraging_effect_model`, we have to show that the **pre-exposure baseline is itself behaving like a baseline**.  A baseline that is silently drifting, full of outliers, or dominated by a single bad day will create a fake reference line for the OFF/ON contrast and inflate effect sizes.

This notebook differs from `indicator_validation.ipynb` (which produced the `baseline_ranges.csv` used by the model) in that **its only purpose is false-positive control**: every single test asks the same question — *would I draw the same conclusion if the labels were random?* — and aggregates the answers into a single verdict per indicator.

## What 'validated' means here

For each of the six indicators we ask:

1. **No within-baseline trend.** A treatment-free window must not have a slope significantly different from zero (the experimental contrast would mis-attribute that slope to 5G).
2. **Stationary mean and variance.** First half ≈ second half.
3. **No single day dominates.** Removing any one day must not change the mean by more than 1 baseline-SD.
4. **Low day-to-day noise.** Coefficient of variation should leave headroom for a treatment effect to register.
5. **Independent of the others.** No two indicators should correlate above |ρ| > 0.9 — otherwise they're double-counting the same biology.
6. **Day-1 effect controlled.** The colony was moved on Apr 13. We test whether dropping day 1 changes any of the above.

## How to read the output

Every test prints a PASS / WARN / FAIL line.  At the very end we collapse those into a single per-indicator verdict.  **An indicator only enters the model if it passes every test or has at most one WARN.**  Any FAIL means the indicator must be excluded or re-defined before running the exposure model.

## Expected results (a priori, what 'all-clean' looks like)

Under the null hypothesis that nothing biological is changing during pre-exposure and the sensor is stable, we expect roughly:

| Test                         | Expected when clean                                                  | Concern when                                                              |
|------------------------------|----------------------------------------------------------------------|---------------------------------------------------------------------------|
| Trend (linear slope)         | permutation p > 0.05 on every indicator                              | p < 0.05 on ≥ 2 indicators (≥ expected 0.05·6 = 0.3 by chance, but ≥ 2 is unusual) |
| Trend after BH correction    | no q < 0.05                                                          | any q < 0.05                                                              |
| Split-half ratio             | |first/second − 1| < 0.30 (i.e. < 30 % drift)                       | > 0.30                                                                    |
| Leave-one-out shift          | mean drift on dropping any day < 1 SD                                | ≥ 1 SD on any day                                                         |
| Lag-1 autocorrelation        | |r₁| < 0.5                                                          | persistent drift visible in time-series plots                              |
| Inter-indicator |ρ|          | < 0.9                                                                | ≥ 0.9 (redundancy)                                                        |
| CUSUM max deviation          | < 2·SD                                                               | a single jump > 2·SD that recovers later (false positive risk)            |

**False-positive control logic.**  Significant trends in a 10-day window are statistically *expected* on roughly 0.05 × 6 ≈ 0.3 indicators by pure chance.  We therefore require **(a) the raw p-value to survive Benjamini-Hochberg correction** (controls FDR across the 6 indicators), **and (b) the trend to be consistent with the leave-one-out and split-half checks**.  A real trend will show up in all three; a single-day outlier or sensor glitch will show up in only one.  *Consistency across the three independent tests is what separates a real effect from a false positive.*

## 0 · Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
rng = np.random.default_rng(20260513)   # reproducible permutations

DATA           = Path("data/multi_day")
CYCLE_ANCHOR   = pd.Timestamp("2026-04-23")
PRE_START      = pd.Timestamp("2026-04-13")
PRE_END        = pd.Timestamp("2026-04-22")
SYSTEM         = 900   # sys 939 has only 2 baseline days; cannot anchor a baseline
N_PERMUTATIONS = 10_000

daily   = pd.read_csv(DATA / "daily_summary.csv",         parse_dates=["date"])
ptracks = pd.read_csv(DATA / "per_track_indicators.csv",  parse_dates=["date", "ts"])
fv      = pd.read_csv(DATA / "flower_visit_summary.csv",  parse_dates=["date"])

print(f"daily : {len(daily):>6,} rows")
print(f"track : {len(ptracks):>6,} rows")
print(f"fv    : {len(fv):>6,} rows")

## 1 · Build the per-day indicator table for the pre-exposure window

Same 6 indicators as `indicator_validation.ipynb` / `5g_foraging_effect_model.ipynb`. Negated where the model expects *higher = more impact* so the model can use a single sign convention.

| Indicator             | Definition                                                         |
|----------------------|--------------------------------------------------------------------|
| `neg_exit_count`     | −1 × number of v3-filtered hive exits per day (activity)            |
| `neg_re_ratio`       | −1 × ratio of returns / exits per day (homing success)              |
| `path_tortuosity`    | per-day median of per-track tortuosity (path complexity)            |
| `ifi_cv`             | inter-foraging-interval coefficient of variation per day            |
| `mean_handling_time_s` | per-day mean dwell on a flower (s)                                |
| `n_distinct_flowers` | per-day count of distinct flower IDs visited                        |

In [ ]:
def in_pre_window(d):
    d = pd.Timestamp(d)
    return (d >= PRE_START) and (d <= PRE_END)

daily_b   = daily  [daily  ["date"].apply(in_pre_window) & (daily  ["system_id"] == SYSTEM)].copy()
ptracks_b = ptracks[ptracks["date"].apply(in_pre_window) & (ptracks["system_id"] == SYSTEM)].copy()
fv_b      = fv     [fv     ["date"].apply(in_pre_window) & (fv     ["system_id"] == SYSTEM)].copy()

# Indicators 1 & 2 — from daily_summary
ind = daily_b[["date", "n_v3", "re_ratio_v3"]].copy()
ind["neg_exit_count"] = -ind["n_v3"]
ind["neg_re_ratio"]   = -ind["re_ratio_v3"]
ind = ind.drop(columns=["n_v3", "re_ratio_v3"])

# Indicator 3 — per-day median tortuosity
tort = ptracks_b.groupby("date")["tortuosity"].median().reset_index(name="path_tortuosity")
ind  = ind.merge(tort, on="date", how="left")

# Indicator 4 — IFI coefficient of variation per day
def _ifi_cv(g):
    ts = g.loc[g["hive_exit_v3"], "ts"].sort_values()
    if len(ts) < 3:
        return np.nan
    iv = ts.diff().dropna().dt.total_seconds()
    return np.nan if iv.mean() == 0 else iv.std() / iv.mean()

ifi = ptracks_b.groupby("date").apply(_ifi_cv).reset_index(name="ifi_cv")
ind = ind.merge(ifi, on="date", how="left")

# Indicators 5 & 6 — flower visits
ind = ind.merge(fv_b[["date", "mean_handling_time_s", "n_distinct_flowers"]],
                on="date", how="left")

ind = ind.sort_values("date").reset_index(drop=True)
INDICATORS = ["neg_exit_count", "neg_re_ratio", "path_tortuosity",
              "ifi_cv", "mean_handling_time_s", "n_distinct_flowers"]

print(f"Pre-exposure days for system {SYSTEM}: {len(ind)}")
print()
print(ind.assign(date=lambda d: d["date"].dt.date).to_string(index=False))

## 2 · Trend test (within-baseline linear regression + permutation)

For each indicator we fit `y = a + b·day_index` and report the slope.  We then permute the day-labels `N_PERMUTATIONS` times to build a null distribution of |slope|.  The permutation p-value is `P(|slope_null| ≥ |slope_observed|)` — this is model-free and works on n = 10 days.  We finally apply Benjamini-Hochberg FDR correction across the 6 indicators.

**Why permutation, not parametric:**  With only 10 baseline days the OLS p-value relies on a normal-residual assumption that we cannot check.  Permutation is exact under the exchangeability null and matches the actual sample size.

**Why FDR correction:**  Six independent t-tests have ≈ 26 % chance of at least one false positive at α = 0.05. Benjamini-Hochberg controls the expected proportion of false discoveries among rejected indicators.

**Interpretation rules:**
- PASS if BH-adjusted q > 0.05.
- WARN if raw p < 0.05 but q > 0.05 (single-test significant; flagged for the leave-one-out cross-check in §4).
- FAIL if q < 0.05.

A FAIL here is the strongest single signal that the baseline is *not* clean — the indicator is trending during a treatment-free window, so its value cannot be used as a reference for the OFF/ON contrast.

In [ ]:
def slope(y):
    """Slope of y vs index 0..n-1, ignoring NaN."""
    y = np.asarray(y, dtype=float)
    mask = ~np.isnan(y); y = y[mask]
    x = np.arange(len(y), dtype=float)
    if len(y) < 3:
        return np.nan
    xm, ym = x.mean(), y.mean()
    sxx = ((x - xm) ** 2).sum()
    if sxx == 0:
        return np.nan
    return float(((x - xm) * (y - ym)).sum() / sxx)

def permutation_p(y, n_perm=N_PERMUTATIONS, rng=rng):
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) < 3:
        return np.nan, np.nan
    obs = abs(slope(y))
    null = np.empty(n_perm)
    for i in range(n_perm):
        null[i] = abs(slope(rng.permutation(y)))
    p = float((null >= obs).mean())
    return obs, p

def benjamini_hochberg(pvals):
    p = np.asarray(pvals, dtype=float)
    n = (~np.isnan(p)).sum()
    order = np.argsort(np.where(np.isnan(p), 1.0, p))
    ranked = np.empty_like(p)
    cur_min = 1.0
    for rank, i in enumerate(order[::-1]):
        if np.isnan(p[i]):
            ranked[i] = np.nan
            continue
        q = p[i] * n / (n - rank)
        cur_min = min(cur_min, q)
        ranked[i] = cur_min
    return ranked

trend_rows = []
for name in INDICATORS:
    s = ind[name].values
    obs_slope, p = permutation_p(s)
    trend_rows.append({
        "indicator": name,
        "slope":     obs_slope,
        "slope_per_day_pct": 100 * obs_slope / abs(np.nanmean(s)) if np.nanmean(s) != 0 else np.nan,
        "perm_p":    p,
    })
trend_df = pd.DataFrame(trend_rows)
trend_df["q_bh"] = benjamini_hochberg(trend_df["perm_p"].values)

def trend_verdict(row):
    if np.isnan(row["perm_p"]):  return "NO DATA"
    if row["q_bh"] < 0.05:        return "FAIL"
    if row["perm_p"] < 0.05:      return "WARN"
    return "PASS"

trend_df["verdict"] = trend_df.apply(trend_verdict, axis=1)
print("=== TREND TEST (permutation, BH-corrected) ===")
print(trend_df.round(4).to_string(index=False))

### How to read this table

- `slope` is the OLS slope (units = indicator-units per day).
- `slope_per_day_pct` is the same slope expressed as % of the indicator's mean — easier to gauge whether the drift is biologically meaningful.
- `perm_p` is the empirical p-value from 10 000 label permutations (model-free).
- `q_bh` is the Benjamini-Hochberg-corrected p-value across the 6 indicators.
- `verdict`: PASS / WARN / FAIL as defined above.

**A WARN here is *not* automatically a false positive.**  It means: this indicator shows a slope you'd see only ~5 % of the time under random shuffling, but with 6 indicators that single hit could happen by chance.  §4 (leave-one-out) and §5 (split-half) are the cross-checks that will tell us which it is.  A trend that survives all three is real.  A trend that disappears when one day is dropped is a single-day outlier (false positive).

## 3 · Split-half reliability (consistency)

Split the 10 baseline days at the midpoint, compare median(first 5) vs median(last 5).  If the relative difference is < 30 %, the indicator's central tendency is stable across the baseline.  This is the simplest possible consistency check; it complements the trend test by being insensitive to slope direction (a U-shaped pattern flagged here would be missed by the linear trend).

**Interpretation rules:**
- PASS if |rel. diff.| < 30 %.
- WARN if 30–50 %.
- FAIL if > 50 %.

In [ ]:
n_days = len(ind)
half   = n_days // 2
split_rows = []
for name in INDICATORS:
    s = ind[name]
    a = s.iloc[:half].dropna()
    b = s.iloc[half:].dropna()
    if len(a) < 2 or len(b) < 2:
        split_rows.append({"indicator": name, "median_first": np.nan,
                            "median_last": np.nan, "rel_diff_pct": np.nan,
                            "verdict": "NO DATA"})
        continue
    m1, m2 = float(a.median()), float(b.median())
    denom = abs((m1 + m2) / 2) or 1.0
    rel  = 100 * abs(m2 - m1) / denom
    if   rel < 30:  v = "PASS"
    elif rel < 50:  v = "WARN"
    else:           v = "FAIL"
    split_rows.append({"indicator": name, "median_first": m1,
                        "median_last": m2, "rel_diff_pct": rel,
                        "verdict": v})
split_df = pd.DataFrame(split_rows)
print("=== SPLIT-HALF CONSISTENCY ===")
print(split_df.round(3).to_string(index=False))

## 4 · Leave-one-out robustness (the false-positive killer)

For each day `d` we recompute the slope on the remaining 9 days.  If removing one specific day collapses the slope (changes the sign or moves it across the WARN/PASS threshold), that day is the source of the apparent trend — i.e. the trend was a *false positive* driven by an outlier, not a real time-varying signal.

We report the **maximum absolute change in slope** when any single day is removed, expressed as a fraction of the full-baseline slope.  A robust trend stays within ±50 %; an outlier-driven trend can swing by > 100 %.

**Interpretation rules per indicator:**
- PASS if no day removal changes the slope by more than 50 % of its full-baseline magnitude AND no day removal moves the verdict from WARN/FAIL to PASS.
- WARN if slope changes by 50–100 %.
- FAIL if removing a single day changes the slope by > 100 % (i.e. flips sign or doubles magnitude).

A WARN/FAIL in §2 combined with a FAIL here is the classic single-day-outlier false-positive pattern.

In [ ]:
loo_rows = []
for name in INDICATORS:
    s_full = ind[name].values
    full_slope = slope(s_full)
    if np.isnan(full_slope) or full_slope == 0:
        loo_rows.append({"indicator": name, "full_slope": full_slope,
                         "max_loo_shift_pct": np.nan, "worst_day": None,
                         "verdict": "NO DATA"})
        continue
    shifts = []
    for i in range(len(s_full)):
        s_drop = np.delete(s_full, i)
        loo_slope = slope(s_drop)
        if np.isnan(loo_slope):
            continue
        shifts.append((i, abs(loo_slope - full_slope) / abs(full_slope)))
    if not shifts:
        loo_rows.append({"indicator": name, "full_slope": full_slope,
                         "max_loo_shift_pct": np.nan, "worst_day": None,
                         "verdict": "NO DATA"}); continue
    worst_i, worst_shift = max(shifts, key=lambda t: t[1])
    pct = 100 * worst_shift
    if   pct < 50:   v = "PASS"
    elif pct < 100:  v = "WARN"
    else:            v = "FAIL"
    loo_rows.append({"indicator": name,
                     "full_slope":  full_slope,
                     "max_loo_shift_pct": pct,
                     "worst_day":   str(ind["date"].iloc[worst_i].date()),
                     "verdict":     v})
loo_df = pd.DataFrame(loo_rows)
print("=== LEAVE-ONE-OUT ROBUSTNESS ===")
print(loo_df.round(3).to_string(index=False))

## 5 · Noise floor (CV) and headroom for treatment effects

If the indicator's day-to-day CV is large, even a sizeable treatment effect will get drowned out in OFF/ON contrasts.  We report CV and the **detectable effect** = SD / mean (=CV) × 1.96 — the smallest mean shift, in %, that a two-sample Welch test could reliably detect with α = 0.05 and a similar sample size.

**Interpretation rules:**
- PASS if CV < 0.5 (treatment effects ≥ 30 % of baseline are detectable).
- WARN if 0.5 ≤ CV < 1.0.
- FAIL if CV ≥ 1.0 (treatment effect would have to exceed the baseline magnitude to be visible).

In [ ]:
cv_rows = []
for name in INDICATORS:
    s = ind[name].dropna()
    if len(s) < 3:
        cv_rows.append({"indicator": name, "mean": np.nan, "sd": np.nan,
                        "cv": np.nan, "min_detectable_pct": np.nan,
                        "verdict": "NO DATA"}); continue
    m, sd = float(s.mean()), float(s.std())
    cv = abs(sd / m) if m != 0 else np.nan
    mde = 100 * (1.96 * sd / abs(m)) if m != 0 else np.nan
    if   cv < 0.5:  v = "PASS"
    elif cv < 1.0:  v = "WARN"
    else:           v = "FAIL"
    cv_rows.append({"indicator": name, "mean": m, "sd": sd,
                    "cv": cv, "min_detectable_pct": mde, "verdict": v})
cv_df = pd.DataFrame(cv_rows)
print("=== NOISE FLOOR (CV) ===")
print(cv_df.round(3).to_string(index=False))

## 6 · Inter-indicator correlation (independence)

Two indicators with |Spearman ρ| > 0.9 carry essentially the same information.  Keeping both inflates the effective number of tests in the model and overstates evidence.  We flag any such pair.

In [ ]:
corr = ind[INDICATORS].corr(method="spearman")
print("=== SPEARMAN CORRELATION ===")
print(corr.round(2).to_string())
print()

redundant = []
for i, a in enumerate(INDICATORS):
    for b in INDICATORS[i + 1:]:
        rho = corr.loc[a, b]
        if abs(rho) > 0.9:
            redundant.append((a, b, rho))

if redundant:
    print("FAIL: redundant indicator pairs (|ρ| > 0.9):")
    for a, b, r in redundant:
        print(f"  {a:25s}  ↔  {b:25s}   ρ = {r:+.2f}")
else:
    print("PASS: no redundant indicator pairs (all |ρ| ≤ 0.9).")

corr_verdict = "FAIL" if redundant else "PASS"

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(INDICATORS))); ax.set_xticklabels(INDICATORS, rotation=45, ha="right")
ax.set_yticks(range(len(INDICATORS))); ax.set_yticklabels(INDICATORS)
for i in range(len(INDICATORS)):
    for j in range(len(INDICATORS)):
        ax.text(j, i, f"{corr.iloc[i, j]:+.2f}", ha="center", va="center",
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black", fontsize=9)
ax.set_title("Inter-indicator Spearman ρ (pre-exposure)")
plt.colorbar(im, ax=ax, label="ρ")
plt.tight_layout(); plt.show()

## 7 · CUSUM (cumulative drift) — catches change-points the linear test misses

A linear-slope test cannot detect a sudden jump that recovers later (e.g. one bad day in the middle).  CUSUM accumulates (value − overall mean) across days; under stationarity it should hover around zero.  A large excursion in either direction signals a change-point or systematic bias.

We report the **maximum absolute CUSUM** in units of the indicator's SD.  Values above 2·SD warrant inspecting the time series plot in §9.

In [ ]:
cusum_rows = []
for name in INDICATORS:
    s = ind[name].dropna().values
    if len(s) < 4:
        cusum_rows.append({"indicator": name, "max_cusum_sd": np.nan,
                            "verdict": "NO DATA"}); continue
    devs = s - s.mean()
    cumsum = np.cumsum(devs)
    max_sd = np.max(np.abs(cumsum)) / (s.std() or 1.0)
    if   max_sd < 2:  v = "PASS"
    elif max_sd < 3:  v = "WARN"
    else:             v = "FAIL"
    cusum_rows.append({"indicator": name, "max_cusum_sd": float(max_sd), "verdict": v})
cusum_df = pd.DataFrame(cusum_rows)
print("=== CUSUM (cumulative deviation from mean) ===")
print(cusum_df.round(3).to_string(index=False))

## 8 · Day-1 sensitivity (acclimatisation check)

The colony was moved into the greenhouse on 2026-04-13.  Bees often take 1–2 days to settle, so day 1 can be an outlier for biological rather than sensor reasons.  We re-run the trend test after dropping day 1 and report whether the verdict changes.  If dropping day 1 *removes* a FAIL, the colony-acclimatisation interpretation is supported and the analysis should be re-run on Apr 14 → Apr 22 (9 days).

In [ ]:
ind_no_d1 = ind.iloc[1:].reset_index(drop=True)
day1_rows = []
for name in INDICATORS:
    s_full = ind[name].values
    s_drop = ind_no_d1[name].values
    obs_full, p_full = permutation_p(s_full)
    obs_drop, p_drop = permutation_p(s_drop)
    day1_rows.append({
        "indicator":      name,
        "p_with_d1":      p_full,
        "p_without_d1":   p_drop,
        "changed_to_pass": (p_full < 0.05) and (p_drop >= 0.05),
    })
day1_df = pd.DataFrame(day1_rows)
print("=== DAY-1 (Apr 13) SENSITIVITY ===")
print(day1_df.round(3).to_string(index=False))
n_recovered = int(day1_df["changed_to_pass"].sum())
print()
if n_recovered:
    print(f"{n_recovered} indicator(s) became non-significant after dropping day 1 →"
          f" acclimatisation effect plausible.")
    print("Recommendation: report both Apr 13→22 (10 days) and Apr 14→22 (9 days)")
    print("baselines in the model; if conclusions agree, day-1 has no leverage.")
else:
    print("No indicator's trend depends on day 1 — baseline starts on a clean foot.")

## 9 · Time-series plot with reference bands

Visual check.  For every indicator we plot the 10 daily values plus three reference lines:
- solid horizontal: baseline mean.
- dashed grey: mean ± 1 SD (the 'typical' band — ≈ 68 % of days should sit inside this).
- dotted red: mean ± 2 SD (the 'outlier' band — by chance only ≈ 5 % of days should sit outside, i.e. expect ~0.5 of 10).

If you see > 1 day outside the 2 SD band, or any clear monotonic drift, cross-check against the §2 / §4 tables.

All six indicators share **no** y-axis on purpose (different units); but every individual indicator can be compared like-for-like across days because it uses one consistent scale.  Per project rules box-plots that compare conditions later will share y-axes — these line plots are not box-plots.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, name in zip(axes.flat, INDICATORS):
    s = ind[name]
    m, sd = float(s.mean()), float(s.std())
    ax.plot(ind["date"], s, marker="o", linewidth=1.5, color="#5B8FD4")
    ax.axhline(m,        color="#444444", linewidth=1)
    ax.axhline(m + sd,   color="#888888", linewidth=0.8, linestyle="--")
    ax.axhline(m - sd,   color="#888888", linewidth=0.8, linestyle="--")
    ax.axhline(m + 2*sd, color="#C04040", linewidth=0.8, linestyle=":")
    ax.axhline(m - 2*sd, color="#C04040", linewidth=0.8, linestyle=":")
    ax.set_title(name); ax.tick_params(axis="x", rotation=30)
fig.suptitle("Pre-exposure indicators (system 900, 2026-04-13 → 2026-04-22)", y=1.02)
plt.tight_layout(); plt.show()

## 10 · Verdict matrix

All six tests rolled into a single PASS / WARN / FAIL per indicator.

**Aggregation rule.**
- **PASS overall** if zero FAIL and at most one WARN.
- **WARN overall** if exactly one FAIL or 2–3 WARNs (acceptable for the model with a noted caveat).
- **FAIL overall** otherwise (the indicator must be excluded from the model, or its definition reviewed).

This is deliberately strict on FAILs (one is enough to disqualify) because a single FAIL means the baseline is *demonstrably* not stable on that dimension.

In [ ]:
verdicts = (
    trend_df.set_index("indicator")["verdict"].rename("trend")
    .to_frame()
    .join(split_df.set_index("indicator")["verdict"].rename("split_half"))
    .join(loo_df  .set_index("indicator")["verdict"].rename("leave_one_out"))
    .join(cv_df   .set_index("indicator")["verdict"].rename("cv"))
    .join(cusum_df.set_index("indicator")["verdict"].rename("cusum"))
)
verdicts["corr"] = corr_verdict

def combine(row):
    counts = row.value_counts()
    n_fail = int(counts.get("FAIL", 0))
    n_warn = int(counts.get("WARN", 0))
    if n_fail == 0 and n_warn <= 1:                    return "PASS"
    if n_fail <= 1 and n_warn <= 3:                    return "WARN"
    return "FAIL"

verdicts["OVERALL"] = verdicts.apply(combine, axis=1)
print("=== VERDICT MATRIX ===")
print(verdicts.to_string())
print()
n_pass = (verdicts["OVERALL"] == "PASS").sum()
n_warn = (verdicts["OVERALL"] == "WARN").sum()
n_fail = (verdicts["OVERALL"] == "FAIL").sum()
print(f"Indicators usable in 5g_foraging_effect_model: PASS={n_pass}, WARN={n_warn}, FAIL={n_fail}")

## 11 · Save the pre-exposure baseline ranges

For every indicator that did **not** FAIL overall we save mean ± 2 SD as the *expected range* during the experimental period.  Days in the OFF condition that fall outside this range are candidate evidence that the colony is not behaving as it did before the experiment started, independent of whether 5G was on at the time.  The model in `5g_foraging_effect_model.ipynb` already consumes this CSV.

We write to `data/multi_day/pre_exposure_ranges.csv` to avoid clobbering the existing `baseline_ranges.csv` produced by `indicator_validation.ipynb` — the two files should be near-identical because they're built from the same data, but they may differ if this notebook excludes a FAIL indicator that the older one kept.

In [ ]:
rng_rows = []
for name in INDICATORS:
    if verdicts.loc[name, "OVERALL"] == "FAIL":
        rng_rows.append({"indicator": name, "baseline_mean": np.nan,
                          "baseline_sd": np.nan, "expected_lo": np.nan,
                          "expected_hi": np.nan,
                          "verdict": "FAIL", "used_in_model": False})
        continue
    s = ind[name].dropna()
    m, sd = float(s.mean()), float(s.std())
    rng_rows.append({"indicator": name, "baseline_mean": m,
                      "baseline_sd": sd, "expected_lo": m - 2*sd,
                      "expected_hi": m + 2*sd,
                      "verdict": verdicts.loc[name, "OVERALL"],
                      "used_in_model": True})
rng_df = pd.DataFrame(rng_rows)
out_path = DATA / "pre_exposure_ranges.csv"
rng_df.to_csv(out_path, index=False)
print(rng_df.round(3).to_string(index=False))
print(f"\nWrote {out_path}")

## 12 · How to act on this verdict — decision tree

**All indicators PASS.**  Use them all in `5g_foraging_effect_model.ipynb`.  Report the pre-exposure window as the reference baseline.  Expected outcome.

**One or two indicators WARN.**  Keep them in the model but report the WARN flags in the model's caveat section.  Verify that any treatment effect on those indicators replicates across the multiple ON/OFF cycles before highlighting it.

**One indicator FAILS.**
1. Look at which sub-test failed in §10.  If `trend` and `leave_one_out` both FAIL on the same day, it's a single-day outlier — check sensor logs and weather for that date.
2. If `trend` FAILS but `leave_one_out` and `split_half` PASS, it's a consistent monotonic drift — likely a real biological signal (colony settling in).  Re-run the entire model on Apr 14 → Apr 22 (drop day 1) and verify conclusions are unchanged.
3. If `cv` FAILS (CV ≥ 1), redefine the indicator: aggregate over a longer window or transform (log, rank) to reduce noise.
4. If `corr` FAILS (two indicators ρ > 0.9), drop one of the pair — keep the one with the lower noise floor.

**Two or more indicators FAIL.**  Stop.  The pre-exposure baseline is not usable as is.  Likely causes (in order of frequency in this dataset):
1. Detection pipeline changed between days (check `download_experiment_data.ipynb` for re-processing artefacts).
2. Sensor uptime gaps — re-run `validation.ipynb` §V.9 / §V.B.4 to confirm coverage.
3. Colony acclimatisation persisting beyond day 1 — extend the pre-exposure window (impossible here, but worth flagging in the methods section).

---

## 13 · How to tell a false positive from a real signal — rules of thumb

Treat the following as a quick checklist when reviewing the tables above:

1. **A single test never decides.**  At α = 0.05 with 6 indicators × 6 sub-tests = 36 chances, ≈ 1.8 spurious failures are expected purely by chance.  Always ask whether the failure replicates in at least one of the other independent tests in §3-§7.
2. **Direction matters.**  If `trend` shows a slope *in the same direction* as the predicted treatment effect (more activity, longer handling, etc.), be more cautious — that's exactly the pattern that would inflate a false positive in the exposure model.  Opposite-direction trends are biologically less worrying for the OFF/ON contrast (they're working against the predicted effect).
3. **Look at the time-series plot in §9.**  A *step* in the middle of the series usually means a sensor or pipeline event; a *gradual ramp* usually means colony adaptation; a *single spike* is an outlier day.  Each has a different remediation in §12.
4. **Cross-reference against `validation.ipynb` §V.B.4 (zero-hour count).**  If a day with a deviant indicator value also has > 4 zero-hours of detections, the indicator failure is downstream of a coverage failure — fix coverage first.
5. **Replication across cycles is the gold standard.**  Anything that survives this notebook should *still* show up in cycle 2 and cycle 3 of the model.  If a baseline `WARN` correlates with a one-off OFF/ON difference that does not repeat, it is almost certainly a false positive driven by exactly this baseline's drift.

---

## 14 · Quick-glance summary string (paste into the model notebook)

In [ ]:
lines = ["Pre-exposure validation (2026-04-13 → 2026-04-22, system 900):"]
for name in INDICATORS:
    lines.append(f"  {name:<25s}  {verdicts.loc[name, 'OVERALL']}")
lines.append(f"  ---")
lines.append(f"  PASS={n_pass}  WARN={n_warn}  FAIL={n_fail}")
summary = "\n".join(lines)
print(summary)

(DATA / "pre_exposure_verdict.txt").write_text(summary)
print(f"\nWrote {DATA / 'pre_exposure_verdict.txt'}")